In [7]:
SAVE_DIR = "Site_Suggestions"
API_URL = "http://192.168.1.23:8191/v1"
BASE_URL = "https://www.ultimate-guitar.com"
FORUM_URL = "https://www.ultimate-guitar.com/forum/forumdisplay.php?f=65"

In [8]:
import os
import re
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, parse_qs, urlencode, urlunparse


os.makedirs(SAVE_DIR, exist_ok=True)

session = requests.Session()


def fetch_html(url: str, max_timeout: int = 60000) -> str:
    post_body = {
        "cmd": "request.get",
        "url": url,
        "maxTimeout": max_timeout
    }

    response = session.post(
        API_URL,
        headers={"Content-Type": "application/json"},
        json=post_body,
        timeout=(max_timeout / 1000) + 30
    )
    response.raise_for_status()

    data = response.json()
    return data["solution"]["response"]


def get_total_pages_from_html(html: str) -> int:
    soup = BeautifulSoup(html, "lxml")

    helper = soup.select_one(".pagination-helper")
    if helper:
        m = re.search(r"Page\s+\d+\s+of\s+(\d+)", helper.get_text(" ", strip=True), re.I)
        if m:
            return int(m.group(1))

    last_link = soup.select_one("ul.pagination li.last a[href]")
    if last_link:
        href = last_link.get("href", "")
        qs = parse_qs(urlparse(href).query)
        if "page" in qs:
            try:
                return int(qs["page"][0])
            except ValueError:
                pass

    max_page = 1
    for m in re.finditer(r"[?&]page=(\d+)", html):
        try:
            max_page = max(max_page, int(m.group(1)))
        except ValueError:
            pass

    return max_page


def extract_thread_links(html: str) -> list[str]:
    soup = BeautifulSoup(html, "lxml")
    links = set()

    for a in soup.select('a[href*="showthread.php?t="]'):
        href = a.get("href")
        if not href:
            continue

        full_url = urljoin(BASE_URL, href)
        m = re.search(r"showthread\.php\?t=(\d+)", full_url)
        if m:
            thread_id = m.group(1)
            links.add(f"{BASE_URL}/forum/showthread.php?t={thread_id}")

    return sorted(links)


def build_forum_page_url(page_num: int) -> str:
    if page_num <= 1:
        return FORUM_URL
    return f"{FORUM_URL}&page={page_num}"


def build_thread_page_url(thread_url: str, page_num: int) -> str:
    parsed = urlparse(thread_url)
    qs = parse_qs(parsed.query)
    qs["page"] = [str(page_num)]
    new_query = urlencode(qs, doseq=True)
    return urlunparse(parsed._replace(query=new_query))


def get_thread_id(url: str) -> str:
    m = re.search(r"[?&]t=(\d+)", url)
    if not m:
        raise ValueError(f"Не удалось извлечь thread id из URL: {url}")
    return m.group(1)


def collect_all_thread_links(limit_forum_pages: int | None = None) -> list[str]:
    first_html = fetch_html(FORUM_URL)
    total_forum_pages = get_total_pages_from_html(first_html)

    if limit_forum_pages is not None:
        total_forum_pages = min(total_forum_pages, limit_forum_pages)

    print(f"Всего страниц раздела: {total_forum_pages}")

    all_thread_links = set()

    for page_num in range(1, total_forum_pages + 1):
        if page_num == 1:
            html = first_html
        else:
            html = fetch_html(build_forum_page_url(page_num))

        page_links = extract_thread_links(html)
        all_thread_links.update(page_links)

        print(f"Страница раздела {page_num}/{total_forum_pages}: найдено тем {len(page_links)}")
        time.sleep(0.2)

    all_thread_links = sorted(all_thread_links)
    print(f"Всего уникальных вопросов найдено: {len(all_thread_links)}")
    return all_thread_links


def download_all_threads(thread_links: list[str], limit_threads: int | None = None):
    if limit_threads is not None:
        thread_links = thread_links[:limit_threads]

    total_threads = len(thread_links)
    print(f"Начинаю скачивать вопросы: {total_threads}")

    for idx, thread_url in enumerate(thread_links, start=1):
        try:
            thread_id = get_thread_id(thread_url)
            thread_dir = os.path.join(SAVE_DIR, f"thread_{thread_id}")
            os.makedirs(thread_dir, exist_ok=True)

            first_html = fetch_html(thread_url)
            total_thread_pages = get_total_pages_from_html(first_html)

            with open(os.path.join(thread_dir, "page_1.html"), "w", encoding="utf-8") as f:
                f.write(first_html)
            print(f"Вопрос {idx}/{total_threads} страница 1 скачалась")

            for page_num in range(2, total_thread_pages + 1):
                page_url = build_thread_page_url(thread_url, page_num)
                html = fetch_html(page_url)

                with open(os.path.join(thread_dir, f"page_{page_num}.html"), "w", encoding="utf-8") as f:
                    f.write(html)

                print(f"Вопрос {idx}/{total_threads} страница {page_num} скачалась")
                time.sleep(0.2)

        except Exception as e:
            print(f"Ошибка при скачивании вопроса {idx}: {thread_url} -> {e}")


# запуск
thread_links = collect_all_thread_links()
download_all_threads(thread_links)
# thread_links = collect_all_thread_links(limit_forum_pages=2)
# download_all_threads(thread_links[:5])


Всего страниц раздела: 114
Страница раздела 1/114: найдено тем 35
Страница раздела 2/114: найдено тем 35
Страница раздела 3/114: найдено тем 35
Страница раздела 4/114: найдено тем 35
Страница раздела 5/114: найдено тем 35
Страница раздела 6/114: найдено тем 35
Страница раздела 7/114: найдено тем 35
Страница раздела 8/114: найдено тем 35
Страница раздела 9/114: найдено тем 35
Страница раздела 10/114: найдено тем 35
Страница раздела 11/114: найдено тем 35
Страница раздела 12/114: найдено тем 35
Страница раздела 13/114: найдено тем 35
Страница раздела 14/114: найдено тем 35
Страница раздела 15/114: найдено тем 35
Страница раздела 16/114: найдено тем 35
Страница раздела 17/114: найдено тем 35
Страница раздела 18/114: найдено тем 35
Страница раздела 19/114: найдено тем 35
Страница раздела 20/114: найдено тем 35
Страница раздела 21/114: найдено тем 35
Страница раздела 22/114: найдено тем 35
Страница раздела 23/114: найдено тем 35
Страница раздела 24/114: найдено тем 35
Страница раздела 25/11